In [2]:
import sys
print(sys.executable)

/Users/zikri/Documents/CS3244/cs3244_group12_imdb_spoilers/.venv/bin/python


In [3]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold

### Locate and load the training split

The training data produced by the outer train-test split is stored under
`data/splits/IMDB_reviews_train.json`.

All experiments below operate only on this development/training set.

In [7]:
PROJECT_ROOT = Path.cwd().parent
# use only the development/training portion produced by the
# stratified train-test split
#
# 20% final test split is deliberately not loaded here because
# it should remain untouched during feature development and
# cross-validation
TRAIN_PATH = (
    PROJECT_ROOT
    / "data"
    / "splits"
    / "IMDB_reviews_train.json"
)

print("Training split:", TRAIN_PATH)
print("Exists:", TRAIN_PATH.exists())

Training split: /Users/zikri/Documents/CS3244/cs3244_group12_imdb_spoilers/data/splits/IMDB_reviews_train.json
Exists: True


In [8]:
train_df = pd.read_json(TRAIN_PATH, lines=True)
print("Training rows:", len(train_df))


Training rows: 458965


## TF-IDF exploration

Before building the reusable cross-validation pipeline, we first examine how
TF-IDF converts review text into numerical features.

A small sample is used here only for inspection and understanding. These
vectors are not used for model evaluation.

The final modelling pipeline will fit a fresh TF-IDF vectorizer only on the
training portion of each cross-validation fold.

## Input data and train-test split

The original cleaned IMDb review dataset is split into an approximately
80% development/training set and 20% final holdout test set before model
development.

The split was created in `02_stratified_train_test_split.ipynb`.

The split preserves approximately the same spoiler/non-spoiler class
distribution in both partitions. Exact duplicate `review_text` values are
also kept within a single partition to prevent identical review text from
appearing in both train and test data.

This produces:

- `data/splits/IMDB_reviews_train.json`
- `data/splits/IMDB_reviews_test.json`

For TF-IDF development and cross-validation, only the training split is
used. The final test split is kept untouched until final model evaluation.

# Inspecting dataset

In [9]:
# checking overall size of the cleaned review dataset
print("Shape:", train_df.shape)


# all available columns in the cleaned dataset
# for our TF-IDF work, the most important ones are:
# - review_text: the actual review content we want to vectorize
# - is_spoiler: the target label used later for stratified splitting
# - movie_id: useful later for movie-disjoint evaluation
print("\nColumns:")
print(train_df.columns.tolist())


# checking whether any reviews are missing text
# TF-IDF expects actual text input, so missing review_text values
# would need to be handled before vectorization
print("\nMissing review_text:")
print(train_df["review_text"].isna().sum())


# counting how many reviews belong to each class
# is_spoiler = True  -> spoiler review
# is_spoiler = False -> non-spoiler review
# this lets us see the amount of class imbalance in the dataset
print("\nClass counts:")
print(train_df["is_spoiler"].value_counts())


# checking proportion of spoiler vs non-spoiler reviews
# we care about this because our later train/test split and k-fold split
# should preserve approximately the same class proportions using stratification
print("\nClass proportions:")
print(train_df["is_spoiler"].value_counts(normalize=True))

Shape: (458965, 7)

Columns:
['review_date', 'movie_id', 'user_id', 'is_spoiler', 'review_text', 'rating', 'review_summary']

Missing review_text:
0

Class counts:
is_spoiler
False    338295
True     120670
Name: count, dtype: int64

Class proportions:
is_spoiler
False    0.737082
True     0.262918
Name: proportion, dtype: float64


## 3. TF-IDF exploration on a small sample

Before applying TF-IDF within cross-validation, we first inspect how
`TfidfVectorizer` converts review text into numerical features.

A small sample is used here only for understanding and debugging.

These vectors are **not** used for final model evaluation.

In [10]:
# display a few review texts together with their spoiler labels
# before turning text into numbers, we want to understand what the raw
# input actually looks like
train_df[["review_text", "is_spoiler"]].head(5)

,review_text,is_spoiler
0,The Shawshank Redemption is without a doubt on...,True
1,I believe that this film is the best story eve...,True
2,"**Yes, there are SPOILERS here**This film has ...",True
3,At the heart of this extraordinary movie is a ...,True
4,In recent years the IMDB top 250 movies has ha...,True


In [11]:
# inspecting one complete review and its label
# it will help us connect the original text to the TF-IDF representation
# that we will create later
print(train_df.iloc[0]["review_text"])
print("\nSpoiler label:", train_df.iloc[0]["is_spoiler"])

The Shawshank Redemption is without a doubt one of the most brilliant movies I have ever seen. Similar to The Green Mile in many respects (and better than it in almost all of them), these two movies have shown us that Stephen King is a master not only of horror but also of prose that shakes the soul and moves the heart. The plot is average, but King did great things with it in his novella that are only furthered by the direction, and the acting is so top-rate it's almost scary.Tim Robbins plays Andy Dufrane, wrongly imprisoned for 20 years for the murder of his wife. The story focuses on Andy's relationship with "Red" Redding (Morgan Freeman, in probably his best role) and his attempts to escape from Shawshank. Bob Gunton is positively evil and frightening as Warden Norton, and there are great performances and cameos all around; the most prominent one being Gil Bellows (late as Billy of Ally McBeal) as Tommy, a fellow inmate of Andy's who suffers under the iron will of Norton.If you ha

In [12]:
# take a very small sample first so we can understand what TF-IDF is doing
# without processing the entire dataset
#
# random_state=42 makes the sample reproducible, so we get the same
# 20 reviews every time we rerun the notebook
sample_df = train_df.sample(
    n=20,
    random_state=42
).reset_index(drop=True)

# show a few sampled reviews and their labels
sample_df[["review_text", "is_spoiler"]].head()

,review_text,is_spoiler
0,"Disclosure is a silly but intriguing thriller,...",True
1,I do enjoy films with a message. V certainly h...,False
2,Now You See Me is a film that is all about obf...,False
3,"Candyman, featuring a career-defining role for...",False
4,(Originally reviewed: 25/01/2017) I have not s...,True


### Fit a simple unigram TF-IDF representation

For the initial exploration, only unigrams are used.

`fit_transform()` performs two operations:

1. learns the vocabulary and IDF statistics from the supplied text;
2. transforms each review into a sparse numerical feature vector.

Each row of the resulting matrix represents one review, while each column
represents one term learned by the vectorizer.

In [13]:
# Creating a basic TF-IDF vectorizer
# lowercase=True:
# converts all words to lowercase so "Movie" and "movie" are treated
# as the same feature

# ngram_range=(1, 1):
# use only individual words (unigrams) for now
# later we might also be experimenting with bigrams such as "the ending"
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 1)
)

# learn the vocabulary and IDF statistics from the sample reviews,
# then convert the reviews into numerical TF-IDF vectors
X = vectorizer.fit_transform(sample_df["review_text"])

In [14]:
# check for the type and shape of the TF-IDF matrix

# rows = number of reviews
# columns = number of unique terms/features learned from the sample
#
# the matrix is sparse because each review only contains a small fraction
# of all possible words in the vocabulary
print("Type:", type(X))
print("Shape:", X.shape)

Type: <class 'scipy.sparse._csr.csr_matrix'>
Shape: (20, 1727)


### Inspect the learned vocabulary

The columns of the TF-IDF matrix correspond to the terms learned from the
training corpus.

The matrix is stored sparsely because each individual review contains only a
small proportion of all terms in the vocabulary.

In [15]:
# get the feature names learned by TF-IDF
# each feature corresponds to one column in X.
feature_names = vectorizer.get_feature_names_out()

print("Number of features:", len(feature_names))
print("\nFirst 50 features:")
print(feature_names[:50])

Number of features: 1727

First 50 features:
['01' '10' '12' '13' '136' '144' '1911' '1939' '1969' '1990s' '2015'
 '2017' '21' '25' '2rated' '85' '88' 'abandon' 'abandoned' 'able' 'about'
 'absolutely' 'accept' 'accident' 'accomplishment' 'accordion' 'accused'
 'ace' 'achievement' 'act' 'acting' 'action' 'actors' 'actresses' 'acts'
 'actual' 'actually' 'address' 'admired' 'admit' 'adolescent' 'adult'
 'adventure' 'affair' 'afforded' 'after' 'again' 'against' 'agent' 'ahead']


### Inspect one review's TF-IDF representation

Only terms appearing in a particular review receive non-zero values in that
review's TF-IDF vector.

The TF-IDF weight reflects both:

- how strongly the term occurs within the document; and
- how informative the term is relative to the documents used to fit the
  vectorizer.

In [16]:
# look at first sampled review
review_index = 0

print("Original review:\n")
print(sample_df.loc[review_index, "review_text"])


# extract the sparse TF-IDF row for this review
row = X[review_index]

# find only the columns with non-zero TF-IDF values
# these correspond to words that actually appear in this review
nonzero_indices = row.nonzero()[1]

print("\nNon-zero TF-IDF features:\n")

for idx in nonzero_indices:
    print(
        feature_names[idx],
        "->",
        row[0, idx]
    )

Original review:

Disclosure is a silly but intriguing thriller, which would have been much better had the screen-writer chosen to concentrate on the sexual harrassment plot thread as opposed to the confusing computer conspiracy mumbo jumbo.The plot of Disclosure revolves around Tom Sanders(Michael Douglas) a fairly stereotypical buisness man, with a high paying job, a loving wife and two children. However on the day he is expecting a promotion he finds that the prestigious position has been given to a woman named Meredith Johnson(Demi Moore) an old flame of his and he begins to suspect his career may be in jeapordy.For this reason he later goes to her office for a buisness discussion, but finds she is more interested in rekindling their romance and after much groping, groaning and close-ups of Demi Moore's wonder bra (but no Sex)he flounces out of her office, declaring he is happily married and has no interest in beginning an affair. Meredith does not respond well to this and the foll

## Train / validation TF-IDF behaviour

TF-IDF must be fitted only on the training portion of a split.

The training text is used to learn:
- the vocabulary,
- document frequencies,
- IDF weights.

The validation text must then be transformed using that same fitted vectorizer.

This avoids allowing validation data to influence the feature representation.

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# use a small subset first so we can verify the workflow quickly
small_df = train_df.sample(
    n=5000,
    random_state=42
)

# split the sample into a temporary train and validation set
#
# stratify=... keeps the spoiler/non-spoiler proportion similar
# in both partitions
small_train, small_val = train_test_split(
    small_df,
    test_size=0.2,
    stratify=small_df["is_spoiler"],
    random_state=42
)

print("Train rows:", len(small_train))
print("Validation rows:", len(small_val))

Train rows: 4000
Validation rows: 1000


### Fit on training, transform validation

`fit_transform()` is used only for the training subset.

The validation subset uses `transform()` so that its representation is based
only on the vocabulary and IDF values learned from the training data.

In [ ]:
# create a fresh TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 1)
)

# fitting only on the training text
# fit_transform() learns the vocabulary and IDF values
# then converts the training reviews into numerical vectors
X_train = vectorizer.fit_transform(
    small_train["review_text"]
)

# we will NOT fit again on validation data
# transform() applies the vocabulary and IDF values learned
# from the training set
X_val = vectorizer.transform(
    small_val["review_text"]
)

y_train = small_train["is_spoiler"]
y_val = small_val["is_spoiler"]

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)

print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)

X_train shape: (4000, 32943)
X_val shape: (1000, 32943)
y_train shape: (4000,)
y_val shape: (1000,)


In [19]:
print(
    "Same number of TF-IDF features:",
    X_train.shape[1] == X_val.shape[1]
)

Same number of TF-IDF features: True


## 5. From one validation split to configurable k-fold cross-validation

The previous section used only one train-validation split.

For model selection, the modelling team requires configurable k-fold
cross-validation.

In k-fold cross-validation:

- the development dataset is divided into `k` folds;
- `k - 1` folds are used for training;
- the remaining fold is used for validation;
- this process repeats until every fold has served as validation once.

The value of `k` is not fixed by the preprocessing pipeline. Downstream
modelling experiments may select any valid value `k >= 2`.

We first demonstrate the basic idea using `StratifiedKFold`.

This preserves approximately the same spoiler/non-spoiler class distribution
across folds.

## Step 1: Understanding configurable stratified k-fold cross-validation

Before introducing the final grouped cross-validation strategy, we first
demonstrate the basic k-fold TF-IDF workflow using `StratifiedKFold`.

The purpose of this section is to establish the core cross-validation behaviour:

1. Split the development data into `k` folds.
2. Use `k - 1` folds for training and one fold for validation.
3. Fit a fresh TF-IDF vectorizer only on the training portion.
4. Transform the validation portion using the fitted vectorizer.
5. Repeat until every fold has been used for validation once.

The number of folds `k` is configurable so downstream modelling experiments
can choose their own valid value.

However, this basic approach does not yet account for duplicate review text.
The next section addresses that additional leakage concern.

At this stage, we are no longer using a single temporary train/validation split. Instead, we use **stratified k-fold cross-validation** on the 80% development/training split created earlier.

The value of `k` is configurable so that the modelling team can choose different numbers of folds depending on their experiments.

For example:

- `k = 5` creates 5 folds.
- In each round, 4 folds are used for training and 1 fold is used for validation.
- Every review appears in the validation portion exactly once across the full cross-validation process.

`StratifiedKFold` is used so that the spoiler/non-spoiler class distribution remains approximately consistent across folds.

For each fold, we create a **fresh TF-IDF vectorizer**.

This is important because TF-IDF learns information from the data, including:

- the vocabulary,
- document frequencies,
- IDF weights.

Therefore, the vectorizer must be fitted only on that fold's training text:

```python
X_train = vectorizer.fit_transform(fold_train["review_text"])

In [20]:
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer

# number of folds is configurable
# Brian / Gerald can later choose values such as 3, 5, 7, 10 adn etc.
k = 5

if k < 2:
    raise ValueError("k must be at least 2.")

# StratifiedKFold tries to preserve the spoiler/non-spoiler
# class distribution within each fold.
skf = StratifiedKFold(
    n_splits=k,
    shuffle=True,
    random_state=42
)

In [21]:
# use a moderate subset first so we can verify the complete
# k-fold workflow before running it over the full training set
cv_sample = train_df.sample(
    n=10000,
    random_state=42
).reset_index(drop=True)

print("Sample size:", len(cv_sample))
print(
    cv_sample["is_spoiler"]
    .value_counts(normalize=True)
)

Sample size: 10000
is_spoiler
False    0.7428
True     0.2572
Name: proportion, dtype: float64


### Demonstrating TF-IDF inside each fold

A new TF-IDF vectorizer must be created for every fold.

This ensures that the vocabulary and IDF statistics for a particular
validation fold are learned only from that fold's training data.

In [22]:
for fold_number, (train_idx, val_idx) in enumerate(
    skf.split(
        cv_sample["review_text"],
        cv_sample["is_spoiler"]
    ),
    start=1
):
    # select the rows belonging to this fold's
    # training and validation portions
    fold_train = cv_sample.iloc[train_idx]
    fold_val = cv_sample.iloc[val_idx]

    # create a fresh TF-IDF vectorizer for this fold
    #
    # a fresh vectorizer is important because each fold must learn
    # its vocabulary and IDF statistics only from its own training data
    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 1)
    )

    # fit TF-IDF only on this fold's training text
    X_train = vectorizer.fit_transform(
        fold_train["review_text"]
    )

    # apply the fitted TF-IDF representation to validation text
    X_val = vectorizer.transform(
        fold_val["review_text"]
    )

    y_train = fold_train["is_spoiler"]
    y_val = fold_val["is_spoiler"]

    print(f"\nFold {fold_number}")
    print("Train rows:", X_train.shape[0])
    print("Validation rows:", X_val.shape[0])
    print("Number of TF-IDF features:", X_train.shape[1])
    print(
        "Train/validation feature count matches:",
        X_train.shape[1] == X_val.shape[1]
    )


Fold 1
Train rows: 8000
Validation rows: 2000
Number of TF-IDF features: 43978
Train/validation feature count matches: True

Fold 2
Train rows: 8000
Validation rows: 2000
Number of TF-IDF features: 44082
Train/validation feature count matches: True

Fold 3
Train rows: 8000
Validation rows: 2000
Number of TF-IDF features: 44213
Train/validation feature count matches: True

Fold 4
Train rows: 8000
Validation rows: 2000
Number of TF-IDF features: 44252
Train/validation feature count matches: True

Fold 5
Train rows: 8000
Validation rows: 2000
Number of TF-IDF features: 44106
Train/validation feature count matches: True


### What this basic k-fold experiment demonstrates

For `k = 5`, the development dataset is divided into five folds.

During each iteration:

- four folds are used for training,
- one fold is used for validation,
- TF-IDF is fitted only on the training portion,
- the validation portion is transformed using the same fitted vectorizer.

This ensures that validation reviews do not contribute to the vocabulary or
IDF statistics learned during that fold.

## 6. Limitation of ordinary StratifiedKFold

Although `StratifiedKFold` preserves the class distribution, it treats every
row independently.

The IMDb dataset contains cases where identical `review_text` values occur
more than once.

Therefore, ordinary `StratifiedKFold` could theoretically place:

- one copy of a review in the training portion; and
- another identical copy in the validation portion.

This would create duplicate-text leakage because the model could be evaluated
on text that it had already encountered during training.

The outer train-test split already avoids this problem by keeping identical
review text together.

The inner cross-validation procedure should preserve the same principle.

Note: This notebook currently demonstrates the k-fold TF-IDF workflow directly. The final reusable implementation will be moved into a Python module under `src/` so that other team members can import and reuse the same preprocessing logic in their modelling notebooks.

## 7. Final cross-validation strategy: StratifiedGroupKFold

The final pipeline therefore uses `StratifiedGroupKFold`.

This combines two requirements:

### Stratification

The spoiler/non-spoiler class distribution is kept approximately balanced
across folds.

### Grouping

Rows with the same exact `review_text` are assigned to the same fold.

Therefore, an identical review cannot appear in both the training and
validation portions of a particular fold.

The grouping variable used here is the review text itself:

```python
groups = reviews["review_text"]

In [28]:
from sklearn.model_selection import StratifiedGroupKFold

k = 5

if k < 2:
    raise ValueError("k must be at least 2.")

sgkf = StratifiedGroupKFold(
    n_splits=k,
    shuffle=True,
    random_state=42
)

### Leakage-safe grouped cross-validation

For every fold:

1. duplicate review text is kept together;
2. TF-IDF is fitted only on the training portion;
3. validation text is transformed using the fitted training vectorizer;
4. duplicate text overlap between training and validation is checked.

In [29]:
for fold_number, (train_idx, val_idx) in enumerate(
    sgkf.split(
        X=cv_sample["review_text"],
        y=cv_sample["is_spoiler"],
        groups=cv_sample["review_text"],
    ),
    start=1
):
    fold_train = cv_sample.iloc[train_idx]
    fold_val = cv_sample.iloc[val_idx]

    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 1)
    )

    X_train = vectorizer.fit_transform(
        fold_train["review_text"]
    )

    X_val = vectorizer.transform(
        fold_val["review_text"]
    )

    # Verify that no exact review text occurs in both sides.
    train_texts = set(fold_train["review_text"])
    val_texts = set(fold_val["review_text"])

    overlap = train_texts.intersection(val_texts)

    print(f"\nFold {fold_number}")
    print("Train:", X_train.shape)
    print("Validation:", X_val.shape)
    print("Duplicate review_text overlap:", len(overlap))


Fold 1
Train: (8000, 43914)
Validation: (2000, 43914)
Duplicate review_text overlap: 0

Fold 2
Train: (8000, 44197)
Validation: (2000, 44197)
Duplicate review_text overlap: 0

Fold 3
Train: (8000, 44080)
Validation: (2000, 44080)
Duplicate review_text overlap: 0

Fold 4
Train: (8000, 44022)
Validation: (2000, 44022)
Duplicate review_text overlap: 0

Fold 5
Train: (8000, 44370)
Validation: (2000, 44370)
Duplicate review_text overlap: 0


## 8. Reusable implementation

The preceding cells demonstrate how the final cross-validation and TF-IDF
pipeline works.

The canonical implementation is placed in `src/text_features.py` so that
downstream modelling notebooks do not need to duplicate this code.

The reusable implementation:

- accepts a configurable value of `k`;
- uses `StratifiedGroupKFold`;
- keeps identical `review_text` values together;
- creates a fresh TF-IDF vectorizer for every fold;
- fits TF-IDF only on the fold's training portion;
- transforms the corresponding validation portion;
- returns the transformed features and labels for downstream modelling.

## 9. Reusable TF-IDF pipeline

This allows downstream modelling notebooks to reuse exactly the same
TF-IDF and cross-validation logic instead of copying preprocessing code.

The number of folds `k` remains configurable for the modelling team.

## 10. Test the reusable pipeline

Finally, we import the reusable implementation and verify that it produces the
expected fold outputs.

Downstream team members can use the same function in their modelling notebooks
and choose their own valid value of `k`.

In [25]:
import sys

sys.path.append(str(PROJECT_ROOT))

from src.text_features import generate_tfidf_folds

In [26]:
cv_sample = train_df.sample(
    n=10000,
    random_state=42
).reset_index(drop=True)

for fold in generate_tfidf_folds(
    cv_sample,
    k=5
):
    print(
        "Fold:",
        fold["fold"],
        "| X_train:",
        fold["X_train"].shape,
        "| X_val:",
        fold["X_val"].shape,
    )

Fold: 1 | X_train: (8000, 43978) | X_val: (2000, 43978)
Fold: 2 | X_train: (8000, 44082) | X_val: (2000, 44082)
Fold: 3 | X_train: (8000, 44213) | X_val: (2000, 44213)
Fold: 4 | X_train: (8000, 44252) | X_val: (2000, 44252)
Fold: 5 | X_train: (8000, 44106) | X_val: (2000, 44106)


### Preventing duplicate-text leakage within cross-validation

The original train-test split keeps identical `review_text` values within the
same partition so that exact duplicate reviews cannot appear in both training
and test data.

The same principle is applied during cross-validation.

`StratifiedGroupKFold` is therefore used instead of ordinary
`StratifiedKFold`.

- **Stratification** attempts to preserve the spoiler/non-spoiler class
  distribution across folds.
- **Grouping by `review_text`** ensures that all rows containing the same
  exact review text remain in the same fold.

This prevents a model from being trained on an exact review and subsequently
being evaluated on an identical copy of that review during validation.